In [4]:
import sys
sys.path.append("..")

In [5]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [6]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [7]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    
    theta_adv = deepcopy(theta_0)
    
    for i in range(theta_0.shape[0]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha

    return theta_adv

In [8]:
def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    thetas = generateThetas(theta_0, alpha)
    if alpha == 0:
        return thetas[0].copy()
    
    Js = np.empty((X_0.shape[0], thetas.shape[0]))
    for i in range(X_0.shape[0]):
         J = RecourseCost(X_0[i], lamb)
         for j, theta in enumerate(thetas):  
            Js[i, j] = J.eval(X_r[i], theta[:-1], np.array([theta[-1]]))
    
    Js_sum = Js.sum(axis=0) 
    Js_sum_maxI = np.argmax(Js_sum)
    theta_adv = thetas[Js_sum_maxI]

    return theta_adv

def generateThetas(theta0 : np.ndarray, alpha):
        # theta0 has bias
        thetas = theta0.copy()
        if alpha == 0:
            return np.array([thetas])
        
        thetas = np.repeat(thetas.reshape(1, theta0.size), (theta0.size * 2) - 1, axis=0)
        thetas_i = 0

        for i in range(theta0.size):
            if i == theta0.size - 1:
                thetas[thetas_i][i] -= alpha
                thetas_i += 1
                break

            thetas[thetas_i][i] += alpha
            thetas_i += 1
            thetas[thetas_i][i] -= alpha
            thetas_i += 1

        return thetas

In [9]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, bias_adv))

def calTheta(xP: np.array, weights: np.array, bias: np.array, alpha: float, methods: str):
    if methods == "L-inf":
        thetaP = calThetaAdv_linf(xP, weights, bias, alpha)
    else:
        thetaP = calThetaAdv_l1(np.hstack((xP, np.array([1]))), np.hstack((weights, bias)), alpha)

    return thetaP[:-1], np.array([thetaP[-1]])

In [ ]:
def evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf_adv = deepcopy(clf)


    for i in tqdm.trange(n, desc=f'[] [] [{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        t_0 = theta_0[i]
        w_0, b_0 = t_0[:-1], t_0[[-1]]
        if alpha != 0:
            w_0_adv, b_0_adv = calTheta(x_r, w_0, b_0, alpha, theta_adv_method)
        else:
            w_0_adv, b_0_adv = w_0.copy(), b_0.copy()

        clf.model.coef_ = w_0.reshape(1,-1)
        clf.model.intercept_ = b_0
        clf_adv.model.coef_ = w_0_adv.reshape(1,-1)
        clf_adv.model.intercept_ = b_0_adv

        J = RecourseCost(x_0, lamb)
        bce_loss, cost, price = J.eval(x_r, w_0_adv, b_0_adv, True)

        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, None, None)

In [165]:
def runCostValidityTradeoff (results: dict, params: dict):
    for adv_method in params['adv_method']:
        for model in params['base_model']:
            for dataset in params['data']:
                model_dataset_name = model + '_' + dataset

                for algorithm in params['algorithms']:
                    for seed in params['seeds']:
                        for v_alpha in params['a_l'][model_dataset_name][algorithm].keys():
                            for v_lamb in params['a_l'][model_dataset_name][algorithm][v_alpha]:
                                data = pd.read_pickle(f"../results/recourse/{model}_{dataset}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                                alpha = data["alpha"].unique().item()
                                lamb = data["lambda"].unique().item()
                                X_0 = np.stack(data["x_0"])
                                X_r = np.stack(data["x_r"])
                                theta_0 = np.stack(data["theta_0"])

                                if params['include_mask'] and algorithm != "L1PSD":
                                    data_l1psd = pd.read_pickle(f"../results/recourse/{model}_{dataset}_L1PSD_0.1_0.1_{seed}.pkl")
                                    mask_i = data_l1psd["i"].to_numpy()
                                    X_0 = X_0[mask_i]
                                    X_r = X_r[mask_i]
                                    theta_0 = theta_0[mask_i]
                                
                                match adv_method:
                                    case "MANY":
                                        res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                                    case "THETA0":
                                        res = evaluate_performance_each(X_0, X_r, theta_0, 0, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                                    case "ONE":
                                        pass
                                    case _:
                                        print(f"{adv_method} has not been implemented yet!")
                                
                                results['adv_method'].append(adv_method)
                                results['model'].append(model)
                                results['dataset'].append(dataset)
                                results['algorithm'].append(algorithm)
                                results['seed'].append(seed)
                                results['alpha'].append(alpha)
                                results['lambda'].append(lamb)
                                results['Cost'].append(res['cost'])
                                results['Current Validity'].append(res['m1_probability'])
                                results['Worst Case Validity'].append(res['wc_probability'])
                                results['BCE Loss'].append(res['loss'])
                                results['J'].append(res['J'])
        
    df_results = pd.DataFrame(results)
    return df_results

In [166]:
params = {}
# 'lr', 'nn'
params['base_model'] = ['lr', 'nn']
# 'synthetic', 'german', 'sba'
params['data'] = ['german', 'sba']
params['seeds'] = range(5)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
# 'ONE', 'MANY', 'LARGESTALPHA', 'SMALLESTALPHA', 'THETA0'
params['adv_method'] = ['MANY', 'THETA0']
params['include_base_model'] = False
params['include_mask'] = True

# params['alphas'] = {params['algorithms'][0] : [0.1],
#             params['algorithms'][1] : [0.1], 
#             params['algorithms'][2] : [0.1], 
#             params['algorithms'][3] : [0.1]}

# params['lambdas'] = dict()
params["a_l"] = dict()
# lr_german
params['a_l']['lr_german'] = {params['algorithms'][0] : {0.1: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5)},
            params['algorithms'][1] : {0.1: [0.5,0.3,0.1,0.04,0.01,0.004,0.001]}, 
            params['algorithms'][2] : {0.1: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.0001, 0.00001]))).round(7)}, 
            params['algorithms'][3] : {0.1: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.0001, 0.00001]))).round(7)}}

# nn_german
params['a_l']['nn_german'] = {params['algorithms'][0] : {0.1: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0, 4.0, 5.0]))).round(5)},
            params['algorithms'][1] : {0.1: [3.0, 0.7, 0.3, 0.1, 0.05, 0.01, 0.001]}, 
            params['algorithms'][2] : {0.1: np.hstack((np.array([1e-7,1e-6,1e-5,1e-4]),np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1))).round(9)}, 
            params['algorithms'][3] : {0.1: np.hstack((np.array([1e-7,1e-6,1e-5,1e-4]),np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1))).round(9)}}

# lr_sba
params['a_l']['lr_sba'] = {params['algorithms'][0] : {0.1: [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.3, 2.6, 2.8, 3.0, 3.1, 3.3, 3.5]},
            params['algorithms'][1] : {0.1: [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.0, 3.5]}, 
            params['algorithms'][2] : {0.1: [0.001, 0.01, 0.08, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1, 2.8, 3.5]}, 
            params['algorithms'][3] : {0.1: [0.001, 0.01, 0.08,0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1, 2.8, 3.5]}}


# nn_sba
params['a_l']['nn_sba'] = {params['algorithms'][0] :{0.1: [0.00001, 0.0001, 0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]},
            params['algorithms'][1] : {0.1: [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]}, 
            params['algorithms'][2] : {0.1: [0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5]}, 
            params['algorithms'][3] : {0.1: [0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5]}}

results = {
    'adv_method': [],
    'model': [],
    'dataset': [],
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
    'BCE Loss': [],
    'J': []
}

df_results = runCostValidityTradeoff(results, params)

[] [] [L1psd] [ seed=4 ] [ α=0.1 ] [ λ=0.001 ]: 100%|██████████| 14/14 [00:00<00:00, 1656.52it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.001 ]: 100%|██████████| 16/16 [00:00<00:00, 2399.83it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.002 ]: 100%|██████████| 16/16 [00:00<00:00, 2480.74it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.003 ]: 100%|██████████| 16/16 [00:00<00:00, 2444.50it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.004 ]: 100%|██████████| 16/16 [00:00<00:00, 2479.45it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.005 ]: 100%|██████████| 16/16 [00:00<00:00, 2441.48it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.006 ]: 100%|██████████| 16/16 [00:00<00:00, 2280.06it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.007 ]: 100%|██████████| 16/16 [00:00<00:00, 2422.00it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.008 ]: 100%|██████████| 16/16 [00:00<00:00, 2483.67it/s]
[] [] [Roarlinf] [ seed=0 ] [ α=0.1 ] [ λ=0.009 ]: 100%|██████████| 16/16 [00:00<00:00, 2440.3

In [167]:
df_results_avg = df_results.groupby(['adv_method', 'model', 'dataset', 'algorithm', 'lambda', 'alpha'], as_index=False).mean()

In [168]:
def plotFigures(df_results_avg, params, should_plot_frontier=False):
    font_family = 'Times New Roman'
    font_color = 'black'
    width, height = 720, 540

    fig = go.Figure()

    if should_plot_frontier:
        for i, alg in enumerate(params["algorithms"]):
            df_alg = df_results_avg.copy()
            df_alg = df_alg[(df_alg['algorithm']==alg)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
            x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
            df_alg = pd.DataFrame({'Algorithm': [f"{alg}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

            fig.add_trace(go.Scatter(
                x = df_alg['Cost'],
                y = df_alg['Worst Case Validity'],
                mode = 'lines+markers' if alg != 'wachter' else 'markers',
                name = f"{alg}",
                showlegend=True,
                # customdata=df_alg['alpha'],
                customdata=df_alg[['alpha', 'lambda']].to_numpy(),
                hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata[0]}<br>lambda: %{customdata[1]}'
            ))

    else:
        c = 0
        for i, alg in enumerate(params["algorithms"]):
            # for alpha in params['alphas']:
            # for alpha in params['alphas'][alg]:
            for alpha in params['a_l'][alg].keys():
                df_alg = df_results_avg.copy()
                df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['alpha']==alpha)]
                x, y = df_alg['Cost'], df_alg['Worst Case Validity']
                df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(alpha={alpha})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'lambda': df_alg['lambda']})

                fig.add_trace(go.Scatter(
                    x = df_alg['Cost'],
                    y = df_alg['Worst Case Validity'],
                    # marker = dict(color=colors[c], size=5),
                    # marker = dict(color=custom_colors[c], size=5),
                    mode = 'lines+markers' if alg != 'wachter' else 'markers',
                    name = f"{alg} (alpha={alpha})",
                    showlegend=True,
                    customdata=df_alg['lambda'],
                    hovertemplate='Cost: %{x}<br>Validity: %{y}<br>lambda: %{customdata}'
                ))
                c+=1

    fig.update_xaxes(
        title=dict(
            text='Cost',
            font=dict(
                family=font_family,
                color=font_color,
                size=25
            )
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey', 
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )


    fig.update_yaxes(
        title=dict(
            text='Worst Case Validity',
            font=dict(
                family=font_family,
                color=font_color,
                size=25
            ), 
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey',
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )


    fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=50,b=25,l=25,r=25),
        title =dict(
            # text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | Average Adversary", 
            text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC ({params['adv_method']}) Adversary", 
            x= 0.5, 
            font=dict(family=font_family, size=20)
            ),
        legend=dict(
            x=0.975, 
            y=0.025, 
            orientation='v',
            xanchor='right',
            font=dict(
                family=font_family,
                color=font_color,
                size=15
                ), 
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='lightgrey',
            borderwidth=1,
            entrywidth=100.5,
            ),
        xaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20,
            ),
            range=[0,25], # german_lr
            # range=[0, 8], # german_nn
            # range=[0,6], # sba
            
        ),
        yaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20
            ),
            # range=[0.2,1.1], # german_lr
            # range=[0.3,1.1], # german_nn
            range=[-0.1,1.1], # sba
        )
    )

    return fig

In [169]:
# should_plot_frontier = False
# fig = plotFigures(df_results_avg, params, should_plot_frontier)
# fig.show()



In [170]:
df_results_avg

,adv_method,model,dataset,algorithm,lambda,alpha,seed,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,MANY,lr,german,Alg1,0.001,0.1,2.0,20.271633,0.999630,0.996757,0.003249,0.023520
1,MANY,lr,german,Alg1,0.002,0.1,2.0,18.012547,0.999071,0.993514,0.006509,0.042534
2,MANY,lr,german,Alg1,0.003,0.1,2.0,16.686437,0.998405,0.990270,0.009779,0.059839
3,MANY,lr,german,Alg1,0.004,0.1,2.0,15.742285,0.997656,0.987027,0.013062,0.076031
4,MANY,lr,german,Alg1,0.005,0.1,2.0,15.007418,0.996836,0.983784,0.016355,0.091392
...,...,...,...,...,...,...,...,...,...,...,...,...
561,THETA0,nn,sba,ROARLInf,1.700,0.1,2.0,0.872460,0.367463,0.367463,1.925499,3.408681
562,THETA0,nn,sba,ROARLInf,1.900,0.1,2.0,0.637850,0.249229,0.249229,2.433652,3.645567
563,THETA0,nn,sba,ROARLInf,2.100,0.1,2.0,0.394047,0.158387,0.158387,3.100318,3.927817
564,THETA0,nn,sba,ROARLInf,2.800,0.1,2.0,0.037850,0.077484,0.077484,4.215694,4.321674


In [171]:
'#636EFA',
'#EF553B',
'#00CC96',
'#AB63FA',
"#8EF1F3",
"#F6B08C",
"#B5FFBE",
"#D7BBF4"

custom_colors = ['#636EFA','#EF553B','#00CC96','#AB63FA']

In [271]:
from plotly.subplots import make_subplots

font_family = 'Times New Roman'
font_color = 'black'
font_size = 18
validity_names = {'THETA0': 'Base Validity', 'MANY': 'Instance-Wise Validity'}

fig_sub = make_subplots(rows = len(params['adv_method']), 
                        cols = 4, 
                        shared_yaxes=True,
                        # x_title="Implementation Cost",
                        # y_title="Worst-Case Validity",
                        subplot_titles= len(params['adv_method']) * ["German(LR)", "German(NN)", "SBA(LR)", "SBA(NN)"],
                        horizontal_spacing=0.025,
                        vertical_spacing=0.1)

for ann in fig_sub['layout']['annotations']:
    ann['font'] = dict(family=font_family,
                        color=font_color,
                        size=18)


for adv_method_index, adv_method in enumerate(params['adv_method']):
    for m_d_index, m_d in enumerate(params['a_l'].keys()):
        model, dataset = m_d.split(sep='_')

        for alg_index, alg in enumerate(params["algorithms"]):
            for alpha in params['a_l'][m_d][alg].keys():
                df_alg = df_results_avg.copy()
                df_alg = df_alg[(df_alg['adv_method'] == adv_method) & (df_alg['model'] == model) & (df_alg['dataset'] == dataset) &(df_alg['algorithm']==alg) & (df_alg['alpha']==alpha)]
                x, y = df_alg['Cost'], df_alg['Worst Case Validity']
                df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(alpha={alpha})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'lambda': df_alg['lambda']})

                fig_sub.add_trace(go.Scatter(
                    x = df_alg['Cost'],
                    y = df_alg['Worst Case Validity'],
                    marker = dict(color=custom_colors[alg_index], size=5),
                    # mode = 'lines+markers' if alg != 'wachter' else 'markers',
                    mode = 'lines+markers',
                    name = f"{alg} (\u03B1={alpha})",
                    showlegend= (m_d_index==1 & adv_method_index==0),
                    customdata=df_alg['lambda'],
                    hovertemplate='Cost: %{x}<br>Validity: %{y}<br>lambda: %{customdata}'
                ), row=adv_method_index+1, col=m_d_index+1)

for i in range(len(params['adv_method'])):
    fig_sub.update_xaxes(range=[0, 15], 
                        showline=True, 
                        mirror=True,
                        linecolor='lightgrey', 
                        gridcolor='lightgrey', 
                        zerolinewidth=1,
                        zerolinecolor='lightgrey',
                        tickfont=dict(family=font_family,
                                    color=font_color,
                                    size=font_size),
                        row=i+1, col=1)
    fig_sub.update_xaxes(range=[0, 6],  
                        showline=True, 
                        mirror=True,
                        linecolor='lightgrey', 
                        gridcolor='lightgrey', 
                        zerolinewidth=1,
                        zerolinecolor='lightgrey',
                        tickfont=dict(family=font_family,
                                    color=font_color,
                                    size=font_size),
                         row=i+1, col=2)
    fig_sub.update_xaxes(range=[0, 6], 
                        showline=True, 
                        mirror=True,
                        linecolor='lightgrey', 
                        gridcolor='lightgrey', 
                        zerolinewidth=1,
                        zerolinecolor='lightgrey',
                        tickfont=dict(family=font_family,
                                    color=font_color,
                                    size=font_size),
                        row=i+1, col=3)
    fig_sub.update_xaxes(range=[0, 6], 
                        showline=True, 
                        mirror=True,
                        linecolor='lightgrey', 
                        gridcolor='lightgrey', 
                        zerolinewidth=1,
                        zerolinecolor='lightgrey',
                        tickfont=dict(family=font_family,
                                    color=font_color,
                                    size=font_size),
                        row=i+1, col=4)

    fig_sub.update_yaxes(title=dict(text=validity_names[params['adv_method'][i]],
                                    font=dict(family=font_family,
                                              color=font_color,
                                              size=20)), 
                         row=i+1, col=1)


fig_sub.update_yaxes(range=[0,1],
                    showline=True, 
                    mirror=True,
                    linecolor='lightgrey', 
                    gridcolor='lightgrey', 
                    zerolinewidth=1,
                    zerolinecolor='lightgrey',
                    tickfont=dict(family=font_family,
                                color=font_color,
                                size=font_size),)

for i in range(4):
    fig_sub.update_xaxes(title=dict(text="Implementation Cost",
                                    font=dict(family=font_family,
                                              color=font_color,
                                              size=20)),
                         row=2,col=i+1)

fig_sub.update_layout(
    width=1400,
    height=len(params['adv_method']) * 400,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation="h",          
        yanchor="bottom",         
        y=-0.18,                  
        xanchor="center",         
        x=0.5,
        font=dict(family=font_family,
                  color=font_color,
                  size=18)),
    # annotations=[dict(text="Implementation Cost",
    #                   x=0.5, y=-0.08,
    #                   xref="paper", yref="paper",
    #                   showarrow=False,
    #                   font=dict(family=font_family,
    #                             color=font_color,
    #                             size=20))]
    )

In [269]:
alpha_escape = '\u03B1'
print(alpha_escape)

α
